# LeetCode 1513 — Number of Substrings With Only 1s bằng Python

Link đề: https://leetcode.com/problems/number-of-substrings-with-only-1s/ · Độ khó: **Medium** · Chủ đề: Math, String

Skeleton LeetCode:

```python
class Solution:
    def numSub(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Phân biệt **chuỗi con (substring)** — liên tiếp, đếm theo vị trí — với tập ký tự hay dãy con.
2. Nắm công thức đếm: một đoạn `L` ký tự liên tiếp có `L·(L+1)/2` chuỗi con, và cách chứng minh bằng "chọn điểm đầu ≤ điểm cuối" hoặc "đếm theo điểm kết thúc".
3. Viết được 4 cách: brute force `O(n³)`/`O(n²)`, **tách nhóm 1 liên tiếp** + công thức, **bộ đếm chạy** một lượt `O(n)`, và cách "Pythonic" với `split`/`groupby`.
4. Hiểu khi nào cần **modulo 10⁹ + 7** và lỗi dùng phép chia số thực `/`.
5. Nhận ra pattern "đếm đoạn con thoả tính chất đóng dưới việc lấy đoạn con" → cộng dồn độ dài đoạn kết thúc tại `i`.

## 2. Đề bài & ví dụ

Cho chuỗi nhị phân `s` (chỉ gồm `'0'` và `'1'`). Trả về số **chuỗi con** (liên tiếp, khác rỗng) mà **mọi ký tự đều là `'1'`**. Kết quả có thể rất lớn → trả về modulo `10⁹ + 7`. Hai chuỗi con ở **vị trí khác nhau** được đếm riêng dù giống nhau về nội dung.

**Ví dụ 1:** `s = "0110111"` → `9` (`"1"` × 5, `"11"` × 3, `"111"` × 1).

**Ví dụ 2:** `s = "101"` → `2`.

**Ví dụ 3:** `s = "111111"` → `21`.

**Ràng buộc:** `1 ≤ len(s) ≤ 10⁵`, `s[i] ∈ {'0','1'}`.

In [1]:
import random
import time
from itertools import groupby
from typing import List

MOD = 10**9 + 7
examples = [("0110111", 9), ("101", 2), ("111111", 21)]
for s, want in examples:
    print(repr(s), "->", want)

'0110111' -> 9
'101' -> 2
'111111' -> 21


## 3. Kiến thức nền

### 3.1 Chuỗi con và slicing

Chuỗi con `s[i:j]` gồm các ký tự từ vị trí `i` đến `j − 1`. Chuỗi độ dài `n` có `n·(n+1)/2` chuỗi con khác rỗng (tính theo vị trí).

In [2]:
s = "abc"
subs = [s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1)]
print(subs, len(subs), "= 3*4/2 =", 3 * 4 // 2)

['a', 'ab', 'abc', 'b', 'bc', 'c'] 6 = 3*4/2 = 6


### 3.2 Công thức `L·(L+1)/2` và phép chia nguyên `//`

`1 + 2 + … + L = L(L+1)/2`. Trong Python, dùng `//` để được **số nguyên**; `/` luôn trả `float`.

In [3]:
for L in [1, 2, 3, 6, 100000]:
    print(L, L * (L + 1) // 2, L * (L + 1) / 2, type(L * (L + 1) / 2).__name__)

1 1 1.0 float
2 3 3.0 float
3 6 6.0 float
6 21 21.0 float
100000 5000050000 5000050000.0 float


### 3.3 `str.split` để tách nhóm

`"0110111".split("0")` → `['', '11', '111']`: mỗi phần tử là một nhóm `'1'` liên tiếp (có thể rỗng).

In [4]:
for s in ["0110111", "101", "111111", "000", "1001"]:
    print(repr(s), "->", s.split("0"), "-> độ dài:", [len(p) for p in s.split("0")])

'0110111' -> ['', '11', '111'] -> độ dài: [0, 2, 3]
'101' -> ['1', '1'] -> độ dài: [1, 1]
'111111' -> ['111111'] -> độ dài: [6]
'000' -> ['', '', '', ''] -> độ dài: [0, 0, 0, 0]
'1001' -> ['1', '', '1'] -> độ dài: [1, 0, 1]


### 3.4 `itertools.groupby`

Gom các phần tử **liên tiếp bằng nhau** thành nhóm: trả về cặp `(khoá, iterator)`.

In [5]:
for key, grp in groupby("0110111"):
    g = list(grp)
    print(key, g, len(g))

0 ['0'] 1
1 ['1', '1'] 2
0 ['0'] 1
1 ['1', '1', '1'] 3


### 3.5 Modulo `10⁹ + 7`

Với `n = 10⁵` ký tự `'1'`, đáp án `≈ 5·10⁹` > `10⁹ + 7` → **bắt buộc** lấy mod. Python không tràn số nên có thể lấy mod ở cuối, nhưng trong C++/Java phải lấy mod (và dùng `long`) trong vòng lặp.

In [6]:
n = 10**5
full = n * (n + 1) // 2
print(full, "> MOD?", full > MOD, "| mod =", full % MOD)

5000050000 > MOD? True | mod = 49965


### 3.6 `enumerate` và biểu thức điều kiện một dòng

`enumerate(s)` cho cặp `(chỉ số, ký tự)`. Biểu thức `a if cond else b` gọn hơn `if/else` nhiều dòng — rất hợp với cập nhật bộ đếm.

In [7]:
cnt = 0
for i, ch in enumerate("1101"):
    cnt = cnt + 1 if ch == "1" else 0
    print(i, ch, cnt)

0 1 1
1 1 2
2 0 0
3 1 1


## 4. Ý tưởng nền tảng / trực giác

Một chuỗi con toàn `'1'` **không thể vượt qua một ký tự `'0'`**. Vậy các chuỗi con hợp lệ nằm gọn trong các **nhóm `'1'` liên tiếp** (tối đa hoá), và các nhóm độc lập với nhau.

Tính tay Ví dụ 1 `"0110111"`:
- Nhóm 1: `"11"` (L = 2) → `2·3/2 = 3` chuỗi con: `"1"`, `"1"`, `"11"`.
- Nhóm 2: `"111"` (L = 3) → `3·4/2 = 6`.
- Tổng `3 + 6 = 9` ✔.

**Cách nhìn thứ hai (đếm theo điểm kết thúc):** nếu tại vị trí `i` đang có `cnt` ký tự `'1'` liên tiếp kết thúc ở `i`, thì có đúng `cnt` chuỗi con toàn `'1'` **kết thúc tại `i`** (bắt đầu ở `i`, `i−1`, …, `i−cnt+1`).

| i | s[i] | cnt (1 liên tiếp kết thúc tại i) | cộng | tổng |
|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 0 |
| 1 | 1 | 1 | 1 | 1 |
| 2 | 1 | 2 | 2 | 3 |
| 3 | 0 | 0 | 0 | 3 |
| 4 | 1 | 1 | 1 | 4 |
| 5 | 1 | 2 | 2 | 6 |
| 6 | 1 | 3 | 3 | 9 |

In [8]:
s = "0110111"
cnt = total = 0
for i, ch in enumerate(s):
    cnt = cnt + 1 if ch == "1" else 0
    total += cnt
    ends = [s[i - k:i + 1] for k in range(cnt)]
    print(f"i={i} s[i]={ch} cnt={cnt} chuỗi con kết thúc tại i: {ends} tổng={total}")

i=0 s[i]=0 cnt=0 chuỗi con kết thúc tại i: [] tổng=0
i=1 s[i]=1 cnt=1 chuỗi con kết thúc tại i: ['1'] tổng=1
i=2 s[i]=1 cnt=2 chuỗi con kết thúc tại i: ['1', '11'] tổng=3
i=3 s[i]=0 cnt=0 chuỗi con kết thúc tại i: [] tổng=3
i=4 s[i]=1 cnt=1 chuỗi con kết thúc tại i: ['1'] tổng=4
i=5 s[i]=1 cnt=2 chuỗi con kết thúc tại i: ['1', '11'] tổng=6
i=6 s[i]=1 cnt=3 chuỗi con kết thúc tại i: ['1', '11', '111'] tổng=9


**Vì sao `L(L+1)/2`?** Trong nhóm `L = 4` ký tự `'1'` (đánh số 0..3), liệt kê chuỗi con theo **độ dài**: dài 1 có 4 cái, dài 2 có 3, dài 3 có 2, dài 4 có 1 → `4 + 3 + 2 + 1 = 10 = 4·5/2`. Bảng tam giác dưới đây vẽ mỗi chuỗi con là một dòng `[đầu..cuối]`.

In [9]:
L = 4
for length in range(1, L + 1):
    items = [f"[{i}..{i + length - 1}]" for i in range(L - length + 1)]
    print(f"dài {length}: {len(items)} cái -> {' '.join(items)}")
print("tổng:", sum(L - t + 1 for t in range(1, L + 1)), "=", L * (L + 1) // 2)

dài 1: 4 cái -> [0..0] [1..1] [2..2] [3..3]
dài 2: 3 cái -> [0..1] [1..2] [2..3]
dài 3: 2 cái -> [0..2] [1..3]
dài 4: 1 cái -> [0..3]
tổng: 10 = 10


## 5. Cách 1 — Brute Force / baseline

**Bản 1 (`O(n³)`):** liệt kê mọi chuỗi con và kiểm tra toàn `'1'`.

**Bản 2 (`O(n²)` worst):** cố định điểm đầu `i`, mở rộng `j` cho tới khi gặp `'0'` thì dừng (vì mọi chuỗi con dài hơn đều chứa `'0'`).

In [10]:
def brute_cubic(s):
    n = len(s)
    cnt = 0
    for i in range(n):
        for j in range(i + 1, n + 1):
            if all(c == "1" for c in s[i:j]):
                cnt += 1
    return cnt % MOD

def brute_extend(s):
    n = len(s)
    cnt = 0
    for i in range(n):
        j = i
        while j < n and s[j] == "1":
            cnt += 1
            j += 1
    return cnt % MOD

for s, want in examples:
    print(repr(s), brute_cubic(s), brute_extend(s), "kỳ vọng", want)

'0110111' 9 9 kỳ vọng 9
'101' 2 2 kỳ vọng 2
'111111' 21 21 kỳ vọng 21


**Độ phức tạp:** bản 1 `O(n³)`; bản 2 `O(n + Σ L²)` — với chuỗi toàn `'1'` dài `10⁵` là `~5·10⁹` bước, quá chậm. Space `O(1)` (bản 1 tạo slice `O(n)`).

**Nút thắt:** bản 2 đếm **từng** chuỗi con một. Nhưng trong một nhóm dài `L`, số chuỗi con chỉ phụ thuộc vào `L` → có công thức đóng.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** đáp án = `Σ_{nhóm} L(L+1)/2` với `L` là độ dài mỗi nhóm `'1'` liên tiếp tối đại.

**Insight 2 (bất biến bộ đếm chạy):** duy trì `cnt` = số `'1'` liên tiếp **kết thúc tại vị trí hiện tại**. Khi đó:
- *nếu* `s[i] == '1'` *thì* `cnt += 1`, và có đúng `cnt` chuỗi con mới kết thúc tại `i` → `ans += cnt`.
- *nếu* `s[i] == '0'` *thì* `cnt = 0` (mọi chuỗi con kết thúc tại `i` đều chứa `'0'`).

Hai cách tương đương vì `1 + 2 + … + L = L(L+1)/2`: cộng dồn `cnt` trong một nhóm chính là công thức tam giác.

## 7. Các cách tiếp theo

### Cách 2 — Tách nhóm + công thức `L(L+1)/2`

1. Duyệt chuỗi, đếm độ dài nhóm `'1'` hiện tại `L`.
2. Gặp `'0'`: cộng `L(L+1)//2`, đặt `L = 0`.
3. **Sau vòng lặp**, cộng nhóm cuối (nếu chuỗi kết thúc bằng `'1'`).

In [11]:
def groups_formula(s):
    ans = L = 0
    for ch in s:
        if ch == "1":
            L += 1
        else:
            ans += L * (L + 1) // 2
            L = 0
    ans += L * (L + 1) // 2           # nhóm cuối!
    return ans % MOD

for s, want in examples:
    print(repr(s), groups_formula(s), "kỳ vọng", want)

'0110111' 9 kỳ vọng 9
'101' 2 kỳ vọng 2
'111111' 21 kỳ vọng 21


In [12]:
class SolutionGroups:
    def numSub(self, s: str) -> int:
        ans = L = 0
        for ch in s:
            if ch == "1":
                L += 1
            else:
                ans += L * (L + 1) // 2
                L = 0
        ans += L * (L + 1) // 2
        return ans % (10**9 + 7)

print([SolutionGroups().numSub(s) for s, _ in examples])

[9, 2, 21]


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 3 — Bộ đếm chạy (đếm theo điểm kết thúc)

Không cần công thức, không cần xử lý "nhóm cuối": mỗi bước cộng `cnt`.

In [13]:
def running_counter(s):
    ans = cnt = 0
    for ch in s:
        cnt = cnt + 1 if ch == "1" else 0
        ans += cnt
    return ans % MOD

for s, want in examples:
    print(repr(s), running_counter(s), "kỳ vọng", want)

'0110111' 9 kỳ vọng 9
'101' 2 kỳ vọng 2
'111111' 21 kỳ vọng 21


In [14]:
class SolutionCounter:
    def numSub(self, s: str) -> int:
        MOD = 10**9 + 7
        ans = cnt = 0
        for ch in s:
            if ch == "1":
                cnt += 1
                ans += cnt
            else:
                cnt = 0
        return ans % MOD

print([SolutionCounter().numSub(s) for s, _ in examples])

[9, 2, 21]


**Độ phức tạp:** Time `O(n)`, Space `O(1)`. Ưu điểm: một nguồn sự thật duy nhất (`cnt`), ít chỗ để quên.

### Cách 4 — Pythonic: `split('0')` hoặc `groupby`

`s.split('0')` cho các nhóm `'1'` (kể cả rỗng, đóng góp 0). Ngắn gọn, chạy nhanh vì vòng lặp ở mức C.

In [15]:
def pythonic_split(s):
    return sum(len(p) * (len(p) + 1) // 2 for p in s.split("0")) % MOD

def pythonic_groupby(s):
    total = 0
    for key, grp in groupby(s):
        if key == "1":
            L = sum(1 for _ in grp)
            total += L * (L + 1) // 2
    return total % MOD

for s, want in examples:
    print(repr(s), pythonic_split(s), pythonic_groupby(s), "kỳ vọng", want)

'0110111' 9 9 kỳ vọng 9
'101' 2 2 kỳ vọng 2
'111111' 21 21 kỳ vọng 21


In [16]:
class SolutionSplit:
    def numSub(self, s: str) -> int:
        return sum(len(p) * (len(p) + 1) // 2 for p in s.split("0")) % (10**9 + 7)

print([SolutionSplit().numSub(s) for s, _ in examples])

[9, 2, 21]


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (list các phần tách ra).

## 8. TRACE từng bước

Trace Cách 2 (tách nhóm) trên `"1101110111"`:

| i | ch | L trước | hành động | ans |
|---|---|---|---|---|
| 0 | 1 | 0 | L=1 | 0 |
| 1 | 1 | 1 | L=2 | 0 |
| 2 | 0 | 2 | ans += 3, L=0 | 3 |
| 3–5 | 111 | 0→3 | L=3 | 3 |
| 6 | 0 | 3 | ans += 6, L=0 | 9 |
| 7–9 | 111 | 0→3 | L=3 | 9 |
| hết | | 3 | ans += 6 (nhóm cuối) | 15 |

In [17]:
def trace_groups(s):
    ans = L = 0
    for i, ch in enumerate(s):
        before = L
        if ch == "1":
            L += 1
            act = f"L={L}"
        else:
            add = L * (L + 1) // 2
            ans += add
            L = 0
            act = f"đóng nhóm dài {before}: ans += {add}, L=0"
        print(f"i={i} ch={ch} L trước={before} -> {act} | ans={ans}")
    add = L * (L + 1) // 2
    ans += add
    print(f"hết chuỗi: nhóm cuối dài {L}: ans += {add} -> ans={ans}")
    return ans % MOD

print("kết quả:", trace_groups("1101110111"))

i=0 ch=1 L trước=0 -> L=1 | ans=0
i=1 ch=1 L trước=1 -> L=2 | ans=0
i=2 ch=0 L trước=2 -> đóng nhóm dài 2: ans += 3, L=0 | ans=3
i=3 ch=1 L trước=0 -> L=1 | ans=3
i=4 ch=1 L trước=1 -> L=2 | ans=3
i=5 ch=1 L trước=2 -> L=3 | ans=3
i=6 ch=0 L trước=3 -> đóng nhóm dài 3: ans += 6, L=0 | ans=9
i=7 ch=1 L trước=0 -> L=1 | ans=9
i=8 ch=1 L trước=1 -> L=2 | ans=9
i=9 ch=1 L trước=2 -> L=3 | ans=9
hết chuỗi: nhóm cuối dài 3: ans += 6 -> ans=15
kết quả: 15


Trace Cách 3 (bộ đếm chạy) trên cùng chuỗi:

In [18]:
def trace_counter(s):
    ans = cnt = 0
    for i, ch in enumerate(s):
        cnt = cnt + 1 if ch == "1" else 0
        ans += cnt
        print(f"i={i} ch={ch} cnt={cnt} ans={ans}")
    return ans % MOD

print("kết quả:", trace_counter("1101110111"))

i=0 ch=1 cnt=1 ans=1
i=1 ch=1 cnt=2 ans=3
i=2 ch=0 cnt=0 ans=3
i=3 ch=1 cnt=1 ans=4
i=4 ch=1 cnt=2 ans=6
i=5 ch=1 cnt=3 ans=9
i=6 ch=0 cnt=0 ans=9
i=7 ch=1 cnt=1 ans=10
i=8 ch=1 cnt=2 ans=12
i=9 ch=1 cnt=3 ans=15
kết quả: 15


## 9. Code nộp LeetCode

Bản chính: bộ đếm chạy (Cách 3).

In [19]:
class Solution:
    def numSub(self, s: str) -> int:
        MOD = 10**9 + 7
        ans = cnt = 0
        for ch in s:
            if ch == "1":
                cnt += 1
                ans += cnt
            else:
                cnt = 0
        return ans % MOD

for s, want in examples:
    assert Solution().numSub(s) == want
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

**Phân hoạch theo điểm kết thúc:** mỗi chuỗi con toàn `'1'` có đúng một điểm kết thúc `i`. Gọi `c_i` = số `'1'` liên tiếp kết thúc tại `i` (0 nếu `s[i] = '0'`). Chuỗi con `s[j..i]` toàn `'1'` ⇔ `i − c_i + 1 ≤ j ≤ i` → có đúng `c_i` chuỗi con hợp lệ kết thúc tại `i`. Vậy đáp án `= Σ c_i`.

**Bất biến:** sau khi xử lý ký tự `i`, biến `cnt = c_i` (quy nạp: `c_i = c_{i−1} + 1` nếu `s[i] = '1'`, ngược lại 0) và `ans = Σ_{t ≤ i} c_t`.

**Cách 2:** trong một nhóm dài `L`, các `c_i` lần lượt là `1, 2, …, L` → tổng `L(L+1)/2`; các nhóm rời nhau và ký tự `'0'` có `c_i = 0`. ∎

Kiểm chứng trực tiếp phân hoạch "theo điểm kết thúc": với mỗi `i`, số chuỗi con toàn `'1'` kết thúc tại `i` (đếm bằng brute force) phải bằng `c_i`.

In [20]:
rng_p = random.Random(10)
for _ in range(200):
    s = "".join(rng_p.choice("01") for _ in range(rng_p.randint(1, 20)))
    c = 0
    for i, ch in enumerate(s):
        c = c + 1 if ch == "1" else 0
        brute_end = sum(1 for j in range(i + 1) if set(s[j:i + 1]) == {"1"})
        assert brute_end == c, (s, i)
print("c_i = số chuỗi con toàn '1' kết thúc tại i: đúng trên 200 chuỗi ngẫu nhiên")

c_i = số chuỗi con toàn '1' kết thúc tại i: đúng trên 200 chuỗi ngẫu nhiên


## 11. Phân tích Time / Space

Gọi `n = len(s) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1a. Brute ba vòng | `O(n³)` | `O(n)` (slice) |
| 1b. Mở rộng tới `'0'` | `O(n + Σ L²)`, worst `O(n²)` | `O(1)` |
| 2. Nhóm + công thức | `O(n)` | `O(1)` |
| 3. Bộ đếm chạy | `O(n)` | `O(1)` |
| 4. `split`/`groupby` | `O(n)` | `O(n)` / `O(1)` |

Bản 1b có average tốt khi chuỗi nhiều `'0'` (nhóm ngắn), worst khi chuỗi toàn `'1'`.

## 12. Bộ test

Ví dụ đề, edge case (một ký tự, toàn `'0'`, toàn `'1'`, xen kẽ, kết thúc bằng `'1'`), random đối chiếu brute (seed cố định), test lớn `10⁵` để kiểm tra modulo.

In [21]:
METHODS = {
    "groups": lambda s: SolutionGroups().numSub(s),
    "counter": lambda s: Solution().numSub(s),
    "split": lambda s: SolutionSplit().numSub(s),
    "groupby": pythonic_groupby,
    "extend": brute_extend,
}
edge = [("0", 0), ("1", 1), ("0000", 0), ("1111", 10), ("10101", 3), ("0101", 2), ("011", 3), ("110", 3)]
for s, want in examples + edge:
    for name, f in METHODS.items():
        got = f(s)
        assert got == want, (name, s, got, want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [22]:
rng = random.Random(1513)
for _ in range(500):
    n = rng.randint(1, 30)
    p = rng.choice([0.2, 0.5, 0.8, 0.95])
    s = "".join("1" if rng.random() < p else "0" for _ in range(n))
    ref = brute_cubic(s)
    for name, f in METHODS.items():
        assert f(s) == ref, (name, s)
print("500 chuỗi ngẫu nhiên: mọi cách khớp brute")

500 chuỗi ngẫu nhiên: mọi cách khớp brute


In [23]:
big1 = "1" * 10**5
expect = (10**5 * (10**5 + 1) // 2) % MOD
rng2 = random.Random(2)
big2 = "".join("1" if rng2.random() < 0.999 else "0" for _ in range(10**5))
for name in ["groups", "counter", "split", "groupby"]:
    t = time.perf_counter()
    r1 = METHODS[name](big1)
    r2 = METHODS[name](big2)
    print(f"{name:8s}: {r1} {r2} ({time.perf_counter() - t:.3f}s)")
    assert r1 == expect and r2 == METHODS["counter"](big2)

groups  : 49965 105859083 (0.006s)
counter : 49965 105859083 (0.009s)
split   : 49965 105859083 (0.000s)


groupby : 49965 105859083 (0.006s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên cộng nhóm cuối cùng

Khi chuỗi **kết thúc bằng `'1'`**, nhóm cuối chưa gặp `'0'` để "đóng".

In [24]:
def wrong_last_group(s):
    ans = L = 0
    for ch in s:
        if ch == "1":
            L += 1
        else:
            ans += L * (L + 1) // 2
            L = 0
    return ans % MOD                                  # SAI: thiếu nhóm cuối

for s, want in examples:
    print(repr(s), "| sai:", wrong_last_group(s), "| đúng:", want)

'0110111' | sai: 3 | đúng: 9
'101' | sai: 1 | đúng: 2
'111111' | sai: 0 | đúng: 21


### Lỗi 2 — Quên modulo

Đúng với chuỗi ngắn, sai với chuỗi dài.

In [25]:
def wrong_no_mod(s):
    ans = cnt = 0
    for ch in s:
        cnt = cnt + 1 if ch == "1" else 0
        ans += cnt
    return ans                                        # SAI: thiếu % MOD

print("sai :", wrong_no_mod("1" * 10**5))
print("đúng:", Solution().numSub("1" * 10**5))

sai : 5000050000
đúng: 49965


### Lỗi 3 — Dùng `/` thay vì `//`

Kết quả là `float`, sai kiểu (và với số rất lớn có thể mất chính xác). Thêm `% MOD` vào float còn cho `float`.

In [26]:
def wrong_float(s):
    return sum(len(p) * (len(p) + 1) / 2 for p in s.split("0")) % MOD   # SAI: '/'

r = wrong_float("0110111")
print("sai :", repr(r), type(r).__name__, "| đúng:", repr(Solution().numSub("0110111")))
# Mất chính xác: số nguyên lớn hơn 2^53 không biểu diễn đúng bằng float
L = 2**30 + 1
print("L(L+1)/2 bằng / rồi int:", int(L * (L + 1) / 2), "| bằng //:", L * (L + 1) // 2)
print("float(2**53 + 1) == 2**53 ?", float(2**53 + 1) == 2**53)

sai : 9.0 float | đúng: 9
L(L+1)/2 bằng / rồi int: 576460753914036224 | bằng //: 576460753914036225
float(2**53 + 1) == 2**53 ? True


### Lỗi 4 — Công thức sai: `L·(L−1)/2` hoặc chỉ đếm `s.count('1')`

`L(L−1)/2` là số cặp (bỏ mất các chuỗi con dài 1); `s.count('1')` chỉ đếm các chuỗi con dài 1.

In [27]:
def wrong_formula(s):
    return sum(len(p) * (len(p) - 1) // 2 for p in s.split("0")) % MOD   # SAI

def wrong_count(s):
    return s.count("1")                                                   # SAI

for s, want in examples:
    print(repr(s), "| L(L-1)/2:", wrong_formula(s), "| count:", wrong_count(s), "| đúng:", want)

'0110111' | L(L-1)/2: 4 | count: 5 | đúng: 9
'101' | L(L-1)/2: 0 | count: 2 | đúng: 2
'111111' | L(L-1)/2: 15 | count: 6 | đúng: 21


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm số đoạn con/chuỗi con mà mọi phần tử thoả P"** (P xét từng phần tử độc lập) → bộ đếm chạy: `cnt = cnt + 1 if P(x) else 0; ans += cnt`.
- Tương đương: tách thành các nhóm tối đại, mỗi nhóm `L(L+1)/2`.
- Đề có "modulo 10⁹ + 7" → kết quả lớn, kiểm tra trường hợp toàn `'1'`.
- Các bài cùng khuôn: LC 1759 (chuỗi con đồng nhất), LC 2348 (mảng con toàn 0), LC 413 (cấp số cộng), LC 2110 (giảm dần mượt), LC 795 (max trong khoảng).

Template tổng quát hoá thành một hàm nhận **vị từ** `pred` — dùng lại cho nhiều bài cùng khuôn:

In [28]:
def count_runs(arr, pred):
    ans = cnt = 0
    for x in arr:
        cnt = cnt + 1 if pred(x) else 0
        ans += cnt
    return ans

print("LC 1513 :", count_runs("0110111", lambda c: c == "1"))
print("LC 2348 :", count_runs([1, 3, 0, 0, 2, 0, 0, 4], lambda x: x == 0))   # ví dụ đề LC 2348 -> 6
print("số chẵn :", count_runs([2, 4, 1, 6, 8, 10], lambda x: x % 2 == 0))

LC 1513 : 9
LC 2348 : 6
số chẵn : 9


## 15. Biến thể của bài

1. **LC 1759 – Count Number of Homogenous Substrings:** mọi ký tự giống nhau (không chỉ `'1'`) → `cnt` reset khi ký tự đổi.
2. **LC 2348 – Number of Zero-Filled Subarrays:** mảng con toàn 0.
3. **Độ dài ≥ k:** nhóm dài `L ≥ k` đóng góp `(L−k+1)(L−k+2)/2`.
4. **Cho phép tối đa một `'0'`:** cửa sổ trượt với số `'0'` ≤ 1, cộng độ dài cửa sổ tại mỗi điểm kết thúc.
5. **LC 413 – Arithmetic Slices:** cùng ý tưởng "đếm theo điểm kết thúc".

In [29]:
# Minh hoạ biến thể 4: số chuỗi con chứa tối đa một '0' (cửa sổ trượt)
def at_most_one_zero(s):
    lo = zeros = ans = 0
    for hi, ch in enumerate(s):
        zeros += ch == "0"
        while zeros > 1:
            zeros -= s[lo] == "0"
            lo += 1
        ans += hi - lo + 1               # số chuỗi con hợp lệ kết thúc tại hi
    return ans

def brute_at_most_one_zero(s):
    n = len(s)
    return sum(1 for i in range(n) for j in range(i + 1, n + 1) if s[i:j].count("0") <= 1)

for s in ["0110111", "101", "0000", "1"]:
    print(repr(s), at_most_one_zero(s), brute_at_most_one_zero(s))

'0110111' 24 24
'101' 6 6
'0000' 4 4
'1' 1 1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Liệt kê / mở rộng tới `'0'` | `O(n³)` / `O(n²)` | `O(1)` | Đối chiếu |
| 2. Nhóm + công thức | `Σ L(L+1)/2` | `O(n)` | `O(1)` | Dễ giải thích bằng toán |
| 3. Bộ đếm chạy | `ans += cnt` | `O(n)` | `O(1)` | Mặc định, ít lỗi nhất |
| 4. `split`/`groupby` | Thư viện chuẩn | `O(n)` | `O(n)` | Viết nhanh, một dòng |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().numSub("1011")` và `Solution().numSub("0")` trả về bao nhiêu?

### Bài 2 (tính tay)
Với `s = "11011110"`, liệt kê độ dài các nhóm `'1'` và tính đáp án bằng công thức.

### Bài 3 (điền TODO)
LC 1759: viết `count_homogenous(s)` — số chuỗi con mà mọi ký tự giống nhau (mod `10⁹+7`).

### Bài 4 (tự viết lại)
Viết `num_sub_groupby(s)` bằng `itertools.groupby` (không dùng `split`).

### Bài 5 (mini challenge)
Viết `count_ones_len_at_least(s, k)`: số chuỗi con toàn `'1'` có độ dài **≥ k**.

In [30]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_homogenous(s):
    # TODO: cnt = cnt + 1 nếu s[i] == s[i-1] ngược lại 1; ans += cnt
    pass

# Bài 4 — TODO
def num_sub_groupby(s):
    # TODO: for key, grp in groupby(s): nếu key == '1' thì cộng L(L+1)//2
    pass

# Bài 5 — TODO
def count_ones_len_at_least(s, k):
    # TODO: mỗi nhóm dài L >= k đóng góp (L-k+1)(L-k+2)//2
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Nhóm `'1'`: `[1, 2]`. (2) `1 + 3`. (3) `4` và `0`.

**Bài 2.** (1) `split('0')` → `['11', '1111', '']`. (2) `3 + 10 + 0`. (3) `13`.

**Bài 3.** (1) Giống Cách 3 nhưng điều kiện là "giống ký tự trước". (2) Ký tự đầu tiên: `cnt = 1`. (3) Đừng quên mod.

**Bài 4.** (1) `groupby(s)` trả `(key, grp)`. (2) Độ dài nhóm: `sum(1 for _ in grp)`. (3) Chỉ cộng khi `key == '1'`.

**Bài 5.** (1) Chuỗi con độ dài `t` trong nhóm dài `L`: `L − t + 1` cái. (2) Cộng `t = k..L`. (3) Tổng = `1 + 2 + … + (L−k+1)`.

## 19. Đáp án tham khảo cho bài tập

In [31]:
# Bài 1, 2
r1, r0 = Solution().numSub("1011"), Solution().numSub("0")
groups = [len(p) for p in "11011110".split("0")]
r2 = sum(L * (L + 1) // 2 for L in groups)
print("Bài 1:", r1, r0, "| Bài 2:", groups, r2)
assert (r1, r0) == (4, 0) and r2 == 13 == Solution().numSub("11011110")

Bài 1: 4 0 | Bài 2: [2, 4, 0] 13


In [32]:
# Bài 3
def count_homogenous_sol(s):
    ans = cnt = 0
    for i, ch in enumerate(s):
        cnt = cnt + 1 if i and ch == s[i - 1] else 1
        ans += cnt
    return ans % MOD

def brute_homogenous(s):
    n = len(s)
    return sum(1 for i in range(n) for j in range(i + 1, n + 1) if len(set(s[i:j])) == 1)

assert count_homogenous_sol("abbcccaa") == 13      # ví dụ LC 1759
assert count_homogenous_sol("xy") == 2 and count_homogenous_sol("zzzzz") == 15

# Bài 4
def num_sub_groupby_sol(s):
    total = 0
    for key, grp in groupby(s):
        if key == "1":
            L = sum(1 for _ in grp)
            total += L * (L + 1) // 2
    return total % MOD

rng = random.Random(4)
for _ in range(300):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 15)))
    assert count_homogenous_sol(s) == brute_homogenous(s)
    b = s.replace("a", "0").replace("b", "1")
    assert num_sub_groupby_sol(b) == brute_cubic(b)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [33]:
# Bài 5
def count_ones_len_at_least_sol(s, k):
    total = 0
    for p in s.split("0"):
        L = len(p)
        if L >= k:
            m = L - k + 1
            total += m * (m + 1) // 2
    return total

def brute_at_least(s, k):
    n = len(s)
    return sum(1 for i in range(n) for j in range(i + k, n + 1) if set(s[i:j]) == {"1"})

for _ in range(300):
    s = "".join("1" if rng.random() < 0.75 else "0" for _ in range(rng.randint(1, 20)))
    k = rng.randint(1, 5)
    assert count_ones_len_at_least_sol(s, k) == brute_at_least(s, k)
assert count_ones_len_at_least_sol("0110111", 1) == 9
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm đoạn con mà mọi phần tử thoả P":**

```python
ans = cnt = 0
for x in arr:
    cnt = cnt + 1 if P(x) else 0     # số phần tử thoả P liên tiếp kết thúc tại x
    ans += cnt                        # = số đoạn con hợp lệ kết thúc tại x
return ans % MOD
```

**Checklist trước khi submit:**
- [ ] Reset `cnt = 0` khi gặp `'0'`.
- [ ] Nếu dùng công thức theo nhóm: cộng **nhóm cuối** sau vòng lặp.
- [ ] Dùng `//`, không dùng `/`.
- [ ] Lấy `% (10**9 + 7)`.
- [ ] Test chuỗi toàn `'1'` dài `10⁵`.
- [ ] Đúng chữ ký: `numSub(self, s: str) -> int`.